In [1]:
# --- 0) Librerías y configuración ---
import arcpy
import os
import re
import json

arcpy.env.overwriteOutput = True

# --- 1) Entradas / salidas ---
bosque_src = r"C:\CFN\SHP\2_etiquetado\bosque_etiquetado.shp"
grillas_fc = r"C:\CFN\SHP\grilla_512x512_areas_select_buff5km_entrenamiento.shp"
gdb_work   = r"C:\Users\diana\Documents\ArcGIS\Projects\CURSO_CF\CURSO_CF.gdb"
out_folder = r"C:\CFN\SHP\2_etiquetado"  # GeoJSON finales (label_<id>.geojson)

os.makedirs(out_folder, exist_ok=True)
arcpy.env.workspace = gdb_work

# --- 2) Validaciones ---
for ruta, nombre in [(bosque_src, "bosque_etiquetado"),
                     (grillas_fc, "grillas_select"),
                     (gdb_work,   "GDB de trabajo")]:
    if not arcpy.Exists(ruta):
        raise FileNotFoundError(f"No existe {nombre}: {ruta}")

# Detectar campo ID en grillas (debe ser 'id')
campo_id = None
for f in arcpy.ListFields(grillas_fc):
    if f.name.lower() == "id":
        campo_id = f.name  # conservar mayúsc/minúsc exactas
        break
if not campo_id:
    raise ValueError("El feature class de grillas no tiene un campo 'id'.")
print(f"Campo ID en grillas: {campo_id}")

# --- 3) Bosque → EPSG:9377 en la GDB ---
sr_9377 = arcpy.SpatialReference(9377)  # MAGNA-SIRGAS 2018 / Origen-Nacional
bosque_9377 = os.path.join(gdb_work, "bosque_9377")
if arcpy.Exists(bosque_9377):
    arcpy.management.Delete(bosque_9377)

desc_bosque = arcpy.Describe(bosque_src)
if not desc_bosque.spatialReference or desc_bosque.spatialReference.factoryCode != 9377:
    arcpy.management.Project(bosque_src, bosque_9377, sr_9377)
else:
    arcpy.management.CopyFeatures(bosque_src, bosque_9377)

# --- 3.1) Crear una versión "limpia" del bosque SIN campos (solo geometría) ---
bosque_limpio = os.path.join(gdb_work, "bosque_limpio")
if arcpy.Exists(bosque_limpio):
    arcpy.management.Delete(bosque_limpio)
arcpy.management.CopyFeatures(bosque_9377, bosque_limpio)

# eliminar todo lo que no sea requerido (así evitamos campos que choquen como 'id')
for f in arcpy.ListFields(bosque_limpio):
    if not f.required:  # conserva OBJECTID y Shape automáticamente
        arcpy.management.DeleteField(bosque_limpio, f.name)
print("✔ Bosque preparado sin columnas extra.")

# --- 3.2) Copiar grillas a la GDB (ya en 9377) ---
grillas_9377 = os.path.join(gdb_work, "grillas_9377")
if arcpy.Exists(grillas_9377):
    arcpy.management.Delete(grillas_9377)
arcpy.management.CopyFeatures(grillas_fc, grillas_9377)

# --- 4) Intersect (grillas primero para conservar su 'id' tal cual) ---
fc_intersect = os.path.join(gdb_work, "bosqueXgrillas_intersect")
if arcpy.Exists(fc_intersect):
    arcpy.management.Delete(fc_intersect)

arcpy.analysis.Intersect(
    in_features=[[grillas_9377, ""], [bosque_limpio, ""]],
    out_feature_class=fc_intersect,
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)
print("✔ Intersect generado (grillas primero).")

# --- 4.1) El campo id que usaremos en el Intersect debe venir SOLO de grillas ---
campos_inter = [f.name for f in arcpy.ListFields(fc_intersect)]
if campo_id in campos_inter:
    campo_id_inter = campo_id      # p.ej. 'id'
elif f"{campo_id}_1" in campos_inter:
    # (no debería pasar porque limpiamos bosque, pero por seguridad)
    campo_id_inter = f"{campo_id}_1"
else:
    raise ValueError(f"No encuentro el campo de id ('{campo_id}' o '{campo_id}_1') en el Intersect.")

print(f"Campo ID en intersect a usar: {campo_id_inter}")

# --- 5) Dissolve por id (una fila por id de grillas) ---
fc_dissolve = os.path.join(gdb_work, "bosque_por_id")
if arcpy.Exists(fc_dissolve):
    arcpy.management.Delete(fc_dissolve)

arcpy.management.Dissolve(
    in_features=fc_intersect,
    out_feature_class=fc_dissolve,
    dissolve_field=campo_id_inter,
    statistics_fields=None,
    multi_part="MULTI_PART",
    unsplit_lines="DISSOLVE_LINES"
)
print("✔ Dissolve por id (de grillas) generado.")

# --- 5.1) Chequeo rápido de cuántos ids quedaron (debería ser ~17) ---
conteo_diss = int(arcpy.management.GetCount(fc_dissolve).getOutput(0))
print(f"Filas (ids) en dissolve: {conteo_diss}")

# --- 6) Exportar cada id: crear FC por id en la GDB y luego GeoJSON ---
def slugify(s, maxlen=100):
    """Nombre seguro para archivos/FC: alfanumérico, guion y guion_bajo."""
    s = str(s).strip()
    s = re.sub(r"[^\w\-]+", "_", s)
    return s[:maxlen] if len(s) > maxlen else s

lyr_diss = "lyr_bosque_por_id"
if arcpy.Exists(lyr_diss):
    arcpy.management.Delete(lyr_diss)
arcpy.management.MakeFeatureLayer(fc_dissolve, lyr_diss)

# Reunir IDs únicos del dissolve
ids = sorted({row[0] for row in arcpy.da.SearchCursor(fc_dissolve, [campo_id_inter])})
print(f"IDs únicos a exportar: {len(ids)} → {ids[:10]}{' ...' if len(ids)>10 else ''}")

for i, gid in enumerate(ids, start=1):
    gid_str = slugify(gid)
    print(f"[{i}/{len(ids)}] Exportando id={gid_str} ...")

    fld = arcpy.AddFieldDelimiters(lyr_diss, campo_id_inter)
    if isinstance(gid, str):
        gid_clean = gid.replace("'", "''")  # duplicar comillas simples si existen
        where = f"{fld} = '{gid_clean}'"
    else:
        where = f"{fld} = {gid}"
    arcpy.management.SelectLayerByAttribute(lyr_diss, "NEW_SELECTION", where)

    # Crear un feature class por id en la GDB
    fc_id_name = f"label_{gid_str}"
    fc_id = os.path.join(gdb_work, fc_id_name)
    if arcpy.Exists(fc_id):
        arcpy.management.Delete(fc_id)
    arcpy.conversion.FeatureClassToFeatureClass(lyr_diss, gdb_work, fc_id_name)

    # Exportar GeoJSON
    n = int(arcpy.management.GetCount(fc_id).getOutput(0))
    out_geojson = os.path.join(out_folder, f"label_{gid_str}.geojson")

    if n > 0:
        arcpy.conversion.FeaturesToJSON(
            in_features=fc_id,
            out_json_file=out_geojson,
            format_json="FORMATTED",
            include_z_values="NO_Z_VALUES",
            include_m_values="NO_M_VALUES",
            geoJSON="GEOJSON"
        )
    else:
        with open(out_geojson, "w", encoding="utf-8") as f:
            json.dump({"type":"FeatureCollection","name":f"label_{gid_str}","features":[]}, f, ensure_ascii=False)

    print(f"   → {out_geojson}")

# Limpiar selección
arcpy.management.SelectLayerByAttribute(lyr_diss, "CLEAR_SELECTION")
print("✅ Proceso terminado.")


Campo ID en grillas: id
✔ Bosque preparado sin columnas extra.
✔ Intersect generado (grillas primero).
Campo ID en intersect a usar: id
✔ Dissolve por id (de grillas) generado.
Filas (ids) en dissolve: 15
IDs únicos a exportar: 15 → [253.0, 307.0, 461.0, 567.0, 631.0, 687.0, 828.0, 943.0, 1000.0, 1001.0] ...
[1/15] Exportando id=253_0 ...
   → C:\CFN\SHP\2_etiquetado\label_253_0.geojson
[2/15] Exportando id=307_0 ...
   → C:\CFN\SHP\2_etiquetado\label_307_0.geojson
[3/15] Exportando id=461_0 ...
   → C:\CFN\SHP\2_etiquetado\label_461_0.geojson
[4/15] Exportando id=567_0 ...
   → C:\CFN\SHP\2_etiquetado\label_567_0.geojson
[5/15] Exportando id=631_0 ...
   → C:\CFN\SHP\2_etiquetado\label_631_0.geojson
[6/15] Exportando id=687_0 ...
   → C:\CFN\SHP\2_etiquetado\label_687_0.geojson
[7/15] Exportando id=828_0 ...
   → C:\CFN\SHP\2_etiquetado\label_828_0.geojson
[8/15] Exportando id=943_0 ...
   → C:\CFN\SHP\2_etiquetado\label_943_0.geojson
[9/15] Exportando id=1000_0 ...
   → C:\CFN\SHP\2_

VERSION 2 CON MAS GUIAS

In [2]:
# --- 0) Librerías y configuración ---
import arcpy
import os
import re
import json
import math

arcpy.env.overwriteOutput = True

# --- 1) Entradas / salidas (ACTUALIZADAS) ---
bosque_src = r"C:\CFN\SHP\2_etiquetado\bosque_etiquetado.shp"
grillas_fc = r"C:\CFN\SHP\grilla_512x512_areas_select_buff5km_entrenamiento.shp"  # <— ACTUALIZADA
gdb_work   = r"C:\Users\diana\Documents\ArcGIS\Projects\CURSO_CF\CURSO_CF.gdb"
out_folder = r"C:\CFN\SHP\3_etiquetado_version3"  # <— ACTUALIZADA

os.makedirs(out_folder, exist_ok=True)
arcpy.env.workspace = gdb_work

# --- 2) Validaciones ---
for ruta, nombre in [(bosque_src, "bosque_etiquetado"),
                     (grillas_fc, "grillas_fc"),
                     (gdb_work,   "GDB de trabajo")]:
    if not arcpy.Exists(ruta):
        raise FileNotFoundError(f"No existe {nombre}: {ruta}")

# Detectar campo ID en grillas (debe ser 'id')
campo_id = None
for f in arcpy.ListFields(grillas_fc):
    if f.name.lower() == "id":
        campo_id = f.name  # conservar mayúsc/minúsc exactas
        break
if not campo_id:
    raise ValueError("El feature class de grillas no tiene un campo 'id'.")
print(f"Campo ID en grillas: {campo_id}")

# --- 3) Preparar SR y proyectar a EPSG:9377 en la GDB ---
sr_9377 = arcpy.SpatialReference(9377)  # MAGNA-SIRGAS 2018 / Origen-Nacional

# Bosque → 9377
bosque_9377 = os.path.join(gdb_work, "bosque_9377")
if arcpy.Exists(bosque_9377):
    arcpy.management.Delete(bosque_9377)

desc_bosque = arcpy.Describe(bosque_src)
if not desc_bosque.spatialReference or desc_bosque.spatialReference.factoryCode != 9377:
    arcpy.management.Project(bosque_src, bosque_9377, sr_9377)
else:
    arcpy.management.CopyFeatures(bosque_src, bosque_9377)

# --- 3.1) Crear una versión "limpia" del bosque SIN campos (solo geometría) ---
bosque_limpio = os.path.join(gdb_work, "bosque_limpio")
if arcpy.Exists(bosque_limpio):
    arcpy.management.Delete(bosque_limpio)
arcpy.management.CopyFeatures(bosque_9377, bosque_limpio)

# eliminar todo lo que no sea requerido (así evitamos campos que choquen como 'id')
for f in arcpy.ListFields(bosque_limpio):
    if not f.required:  # conserva OBJECTID y Shape automáticamente
        arcpy.management.DeleteField(bosque_limpio, f.name)
print("✔ Bosque preparado sin columnas extra.")

# --- 3.2) Grillas → 9377 (copiar o proyectar según corresponda) ---
grillas_9377 = os.path.join(gdb_work, "grillas_9377")
if arcpy.Exists(grillas_9377):
    arcpy.management.Delete(grillas_9377)

desc_grillas = arcpy.Describe(grillas_fc)
if not desc_grillas.spatialReference or desc_grillas.spatialReference.factoryCode != 9377:
    # Si la grilla no está en 9377, la proyectamos
    arcpy.management.Project(grillas_fc, grillas_9377, sr_9377)
else:
    # Si ya está en 9377, solo la copiamos a la GDB
    arcpy.management.CopyFeatures(grillas_fc, grillas_9377)

# --- 4) Intersect (grillas primero para conservar su 'id' tal cual) ---
fc_intersect = os.path.join(gdb_work, "bosqueXgrillas_intersect")
if arcpy.Exists(fc_intersect):
    arcpy.management.Delete(fc_intersect)

arcpy.analysis.Intersect(
    in_features=[[grillas_9377, ""], [bosque_limpio, ""]],
    out_feature_class=fc_intersect,
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="INPUT"
)
print("✔ Intersect generado (grillas primero).")

# --- 4.1) El campo id que usaremos en el Intersect debe venir SOLO de grillas ---
campos_inter = [f.name for f in arcpy.ListFields(fc_intersect)]
if campo_id in campos_inter:
    campo_id_inter = campo_id      # p.ej. 'id'
elif f"{campo_id}_1" in campos_inter:
    # (no debería pasar porque limpiamos bosque, pero por seguridad)
    campo_id_inter = f"{campo_id}_1"
else:
    raise ValueError(f"No encuentro el campo de id ('{campo_id}' o '{campo_id}_1') en el Intersect.")

print(f"Campo ID en intersect a usar: {campo_id_inter}")

# --- 5) Dissolve por id (una fila por id de grillas) ---
fc_dissolve = os.path.join(gdb_work, "bosque_por_id")
if arcpy.Exists(fc_dissolve):
    arcpy.management.Delete(fc_dissolve)

arcpy.management.Dissolve(
    in_features=fc_intersect,
    out_feature_class=fc_dissolve,
    dissolve_field=campo_id_inter,
    statistics_fields=None,
    multi_part="MULTI_PART",
    unsplit_lines="DISSOLVE_LINES"
)
print("✔ Dissolve por id (de grillas) generado.")

# --- 5.1) Chequeo rápido ---
conteo_diss = int(arcpy.management.GetCount(fc_dissolve).getOutput(0))
print(f"Filas (ids) en dissolve: {conteo_diss}")

# --- 6) Funciones auxiliares ---
def normalize_id(val):
    """Devuelve el id como string sin .0 cuando es entero."""
    if isinstance(val, (int,)) or (isinstance(val, float) and math.isfinite(val) and val.is_integer()):
        return str(int(val))
    return str(val)

def slugify(s, maxlen=100):
    """Nombre seguro para archivos/FC: alfanumérico, guiones y guion_bajo."""
    s = str(s).strip()
    s = re.sub(r"[^\w\-]+", "_", s)
    return s[:maxlen] if len(s) > maxlen else s

# --- 7) Exportar cada id: crear FC por id en la GDB y luego GeoJSON ---
lyr_diss = "lyr_bosque_por_id"
if arcpy.Exists(lyr_diss):
    arcpy.management.Delete(lyr_diss)
arcpy.management.MakeFeatureLayer(fc_dissolve, lyr_diss)

# Reunir IDs únicos del dissolve
ids = sorted({row[0] for row in arcpy.da.SearchCursor(fc_dissolve, [campo_id_inter])})
print(f"IDs únicos a exportar: {len(ids)} → {ids[:10]}{' ...' if len(ids)>10 else ''}")

for i, gid in enumerate(ids, start=1):
    gid_str = slugify(normalize_id(gid))
    print(f"[{i}/{len(ids)}] Exportando id={gid_str} ...")

    fld = arcpy.AddFieldDelimiters(lyr_diss, campo_id_inter)
    if isinstance(gid, str):
        gid_clean = gid.replace("'", "''")  # duplicar comillas simples si existen
        where = f"{fld} = '{gid_clean}'"
    else:
        where = f"{fld} = {gid}"
    arcpy.management.SelectLayerByAttribute(lyr_diss, "NEW_SELECTION", where)

    # Crear un feature class por id en la GDB
    fc_id_name = f"label_{gid_str}"
    fc_id = os.path.join(gdb_work, fc_id_name)
    if arcpy.Exists(fc_id):
        arcpy.management.Delete(fc_id)
    arcpy.conversion.FeatureClassToFeatureClass(lyr_diss, gdb_work, fc_id_name)

    # Exportar GeoJSON
    n = int(arcpy.management.GetCount(fc_id).getOutput(0))
    out_geojson = os.path.join(out_folder, f"label_{gid_str}.geojson")

    if n > 0:
        arcpy.conversion.FeaturesToJSON(
            in_features=fc_id,
            out_json_file=out_geojson,
            format_json="FORMATTED",
            include_z_values="NO_Z_VALUES",
            include_m_values="NO_M_VALUES",
            geoJSON="GEOJSON"
        )
    else:
        with open(out_geojson, "w", encoding="utf-8") as f:
            json.dump({"type":"FeatureCollection","name":f"label_{gid_str}","features":[]}, f, ensure_ascii=False)

    print(f"   → {out_geojson}")

# Limpiar selección
arcpy.management.SelectLayerByAttribute(lyr_diss, "CLEAR_SELECTION")
print("✅ Proceso terminado.")


Campo ID en grillas: id
✔ Bosque preparado sin columnas extra.
✔ Intersect generado (grillas primero).
Campo ID en intersect a usar: id
✔ Dissolve por id (de grillas) generado.
Filas (ids) en dissolve: 39
IDs únicos a exportar: 39 → [195.0, 203.0, 253.0, 294.0, 307.0, 402.0, 461.0, 566.0, 567.0, 611.0] ...
[1/39] Exportando id=195 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_195.geojson
[2/39] Exportando id=203 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_203.geojson
[3/39] Exportando id=253 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_253.geojson
[4/39] Exportando id=294 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_294.geojson
[5/39] Exportando id=307 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_307.geojson
[6/39] Exportando id=402 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_402.geojson
[7/39] Exportando id=461 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_461.geojson
[8/39] Exportando id=566 ...
   → C:\CFN\SHP\3_etiquetado_version3\label_566.geojson
[9/39] Expor